# PySpark ETL Example

Converted from `etl_sample.py` to notebook cells.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, when, lit

In [2]:
spark = (
    SparkSession.builder
    .appName("pyspark-etl-example")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate()
)

In [3]:
src_csv = r"d:\Python\pyspark\sample.csv"
out_parquet = r"d:\Python\pyspark\output_parquet"

In [4]:
df = (
    spark.read.option("header", True)
              .option("inferSchema", True)
              .csv(src_csv)
)
df.show(5)

+--------+----+------+
|category|name|amount|
+--------+----+------+
|       A| foo|    10|
|       B| bar|    -5|
|       A| baz|    20|
|       C| qux|  NULL|
+--------+----+------+



In [8]:
df_clean = (
    df.withColumn("name", trim(col("name")))
      .withColumn("amount", col("amount").cast("double"))
      .filter(col("amount").isNotNull())
      .withColumn("status", when(col("amount") > 0, lit("ok")).otherwise(lit("bad")))
)
df_clean.show(5)

+--------+----+------+------+
|category|name|amount|status|
+--------+----+------+------+
|       A| foo|  10.0|    ok|
|       B| bar|  -5.0|   bad|
|       A| baz|  20.0|    ok|
+--------+----+------+------+



In [ ]:
agg = (
    df_clean.groupBy("category")
            .agg({"amount": "sum", "*": "count"})
            .withColumnRenamed("sum(amount)", "total_amount")
            .withColumnRenamed("count(1)", "row_count")
)
agg.show(truncate=False)

In [ ]:
agg.write.mode("overwrite").parquet(out_parquet)
print("Result rows:", agg.count())

In [ ]:
spark.stop()